# Q3 baseline: the co-author seed and first journal entry

The question

- does having a co-author who already published in a journal raise the chance of entering that journal for the first time

The unit is one opportunity: one row per (author, journal, year) in which the author was active and had not previously published in the journal. `C` marks an earlier collaborator who had already published in that journal before year t. `F` marks the author's first entry in year t. The topic-adjusted comparison is the main Q3 analysis, so the notebook reports two stages

1. the naive contrast `F ~ C`, computable today and explicitly unadjusted
2. a provisional adjustment `F ~ C + T` with Pierre's rolling topic fit, marked provisional because the export is a local run pending Pierre's confirmation, the treatment of missing `T` is open, and the frozen-profile sensitivity has no common anchor for `C = 0`

The topic-adjusted result remains provisional until the threshold-free export and the treatment of missing `T` are agreed. A separate Prolog implementation matched `C`, `F`, and the ride flag on all 6,422,558 rows. Variant A is the main opportunity set and B the sensitivity, so this notebook uses variant A only.

Inputs, both local in `../data/` and not in git

- `event_table_python_v0_oppA.csv`, 6,422,558 rows
- `event_table_topicmatch_v5.csv`, the same rows with `topic_match` added, Pierre's v5 export (sha256 8a9e8a92, 497,765,187 bytes): threshold removed, journal profiles from the full corpus, adapter active. An independent local regeneration of the same pipeline matches it on every status and count column and on all 1,106,356 topic values to 4.7e-7.

## Step 1: load and align

- both files carry the same rows in the same order, which is asserted on the key columns before the topic column is attached
- `F` is read off `entering_work_id`, filled exactly on entry rows

In [ ]:
# Fetch verified shared inputs; the existing analysis paths stay the same.
import sys
from pathlib import Path
project_root = Path.cwd().resolve().parent
if not (project_root / "data-manifest.json").is_file():
    raise RuntimeError("Run this notebook with its working directory set to nets/.")
sys.path.insert(0, str(project_root))
from project_data import ensure_data
ensure_data("q3", root=project_root)

import numpy as np, pandas as pd

usecols = ["author_id", "journal_id", "t", "n_prior_papers", "coauthor_seed",
           "first_entry_ride", "entering_work_id"]
ev = pd.read_csv("../data/event_table_python_v0_oppA.csv", usecols=usecols, low_memory=False,
                 dtype={"t": "int16", "n_prior_papers": "int32", "coauthor_seed": "int8",
                        "first_entry_ride": "int8", "entering_work_id": "str"})
tm = pd.read_csv("../data/event_table_topicmatch_v5.csv", low_memory=False,
                 usecols=["author_id", "journal_id", "t", "topic_match", "tm_status", "profile_cutoff"],
                 dtype={"t": "int16"})

# same rows in the same order, otherwise attaching by position would silently mix rows
assert len(ev) == len(tm)
assert (ev["author_id"].values == tm["author_id"].values).all()
assert (ev["journal_id"].values == tm["journal_id"].values).all()
assert (ev["t"].values == tm["t"].values).all()
ev["T"] = tm["topic_match"].values
ev["tm_status"] = tm["tm_status"].values

# the status column must explain the fill exactly, and every profile must stop before t
assert ((tm["tm_status"] == "ok") == tm["topic_match"].notna()).all()
assert (tm.loc[tm.tm_status == "ok", "profile_cutoff"] < tm.loc[tm.tm_status == "ok", "t"]).all()
del tm

ev["F"] = ev["entering_work_id"].notna().astype("int8")
C = ev["coauthor_seed"].values
F = ev["F"].values
print(f"{len(ev):,} rows, {int(F.sum()):,} entries, {(C==1).sum():,} rows with a seed")

## Step 2: check against the build

The event table build printed its own headline counts. Rebuilding them here from the loaded frame confirms nothing was lost or doubled on the way in.

In [ ]:
assert int(F.sum()) == 96819                       # one entry row per (author, journal) pair in the corpus
assert int((C == 1).sum()) == 19035                # rows where the seed predates t
assert int(F[C == 1].sum()) == 1784                # entries that happened with a seed in place
assert int(ev["first_entry_ride"].sum()) == 756    # entries with the qualifying seed co-author on the paper
# T coverage changes between export versions, the fill contract is asserted at load time instead
print(f"all four event counts match the build, T exists on {ev['T'].notna().sum():,} rows in this export")

## Step 3: the naive contrast

- entry rate with a seed against entry rate without, nothing held constant
- this number is confounded by design, an author with a seed is better connected and probably closer to the journal's topics, so read it as a crude association, not an effect in either direction
- the interval resamples whole authors, because one author contributes many related rows and row-level intervals would be too narrow
- two ratios are reported, as pre-registered: Q3_all counts every first entry, Q3_ind counts only entries without the seed co-author on the entering paper, each against the unseeded entry rate

In [ ]:
p1, p0 = F[C == 1].mean(), F[C == 0].mean()
print(f"entry rate with a seed    {p1:.4%}  ({int(F[C==1].sum()):,} of {(C==1).sum():,})")
print(f"entry rate without        {p0:.4%}  ({int(F[C==0].sum()):,} of {(C==0).sum():,})")
print(f"naive rate ratio          {p1/p0:.2f}")

# author level counts once, then the bootstrap only touches these four arrays
codes, authors = pd.factorize(ev["author_id"])
nA = len(authors)
n1 = np.bincount(codes[C == 1], minlength=nA)
e1 = np.bincount(codes[(C == 1) & (F == 1)], minlength=nA)
n0 = np.bincount(codes[C == 0], minlength=nA)
e0 = np.bincount(codes[(C == 0) & (F == 1)], minlength=nA)

rng = np.random.default_rng(31)
reps = []
for _ in range(2000):
    idx = rng.integers(0, nA, nA)   # draw authors with replacement, rows follow their author
    a, b, c, d = e1[idx].sum(), n1[idx].sum(), e0[idx].sum(), n0[idx].sum()
    if b and c and d:
        reps.append((a / b) / (c / d))
lo, hi = np.percentile(reps, [2.5, 97.5])
print(f"author-clustered 95% CI   [{lo:.2f}, {hi:.2f}]  ({len(reps)} bootstrap draws)")

# Q3_ind, the pre-registered headline: entries without the seed co-author on the paper over the unseeded entry rate
ride_flag = ev["first_entry_ride"].values
Find = ((F == 1) & (ride_flag == 0)).astype("int8")
print(f"\nQ3_ind crude              {Find[C == 1].mean() / F[C == 0].mean():.2f}  "
      f"({int(Find[C == 1].sum()):,} independent entries on {(C == 1).sum():,} seeded rows)")
i1 = np.bincount(codes[(C == 1) & (Find == 1)], minlength=nA)
rng_ind = np.random.default_rng(32)
reps_ind = []
for _ in range(2000):
    idx = rng_ind.integers(0, nA, nA)
    a, b, c_, d = i1[idx].sum(), n1[idx].sum(), e0[idx].sum(), n0[idx].sum()
    if b and c_ and d:
        reps_ind.append((a / b) / (c_ / d))
lo_i, hi_i = np.percentile(reps_ind, [2.5, 97.5])
print(f"author-clustered 95% CI   [{lo_i:.2f}, {hi_i:.2f}]")

## Step 4: ride against independent, inside the seeded entries

Of the entries that happened with a seed in place, some carry the seed co-author on the entering paper itself and some do not. The split matters because riding along and entering independently are different mechanisms, and only the seeded entries can show it.

In [ ]:
rides = int(ev.loc[(C == 1) & (F == 1), "first_entry_ride"].sum())
seeded_entries = int(F[C == 1].sum())
print(f"seeded entries {seeded_entries:,}")
print(f"  with the seed co-author on the entering paper   {rides:,}  ({rides/seeded_entries:.0%})")
print(f"  without, so independent of that person          {seeded_entries-rides:,}  ({1-rides/seeded_entries:.0%})")

## Step 5: the contrast by year

The corpus starts in 2015, so no observed co-author relation can predate that year. Seeded cells only reach a useful size around 2021. Earlier estimates are based on very few entries and should not be read as a trend.

In [ ]:
g = ev.groupby("t", observed=True)
yr = pd.DataFrame({
    "seeded_rows": g.apply(lambda x: int((x.coauthor_seed == 1).sum()), include_groups=False),
    "seeded_entries": g.apply(lambda x: int(((x.coauthor_seed == 1) & (x.F == 1)).sum()), include_groups=False),
    "other_rows": g.apply(lambda x: int((x.coauthor_seed == 0).sum()), include_groups=False),
    "other_entries": g.apply(lambda x: int(((x.coauthor_seed == 0) & (x.F == 1)).sum()), include_groups=False),
})
yr["rate_ratio"] = (yr.seeded_entries / yr.seeded_rows) / (yr.other_entries / yr.other_rows)
print(yr.to_string(float_format=lambda v: f"{v:.2f}"))
print("\nthe cells before 2021 hold single digit seeded entries, their ratios are noise, not a trend")

## Step 6: what T covers, with the reasons named

`topic_match` is the rolling author profile against the journal profile, 0 to 1, from the threshold-free run. `profile_cutoff` sits strictly before t on every filled row, and every empty cell names its reason in `tm_status`

- the whole-period paper threshold is removed, so eligibility no longer looks at future productivity, and journal profiles now build from every embeddable paper
- the remaining gaps are structural, most entrants have no pre-t paper at all, and a smaller block has papers without usable abstracts or journals without a profile before t

In [ ]:
has = ev["T"].notna().values
print(f"T exists on {has.sum():,} rows ({has.mean():.1%})")
print(f"  on entry rows            {has[F==1].mean():.1%}")
print(f"  on seeded rows           {has[C==1].mean():.1%}")
print(f"  on seeded entries        {has[(C==1)&(F==1)].mean():.1%}  ({int(has[(C==1)&(F==1)].sum()):,} of {int(((C==1)&(F==1)).sum()):,})")
print(f"  on rides                 {has[ev.first_entry_ride.values==1].mean():.1%}")
print("\nwhy the rest is empty, all rows")
print(ev["tm_status"].value_counts().to_string())
print("\non entry rows")
print(ev.loc[F == 1, "tm_status"].value_counts().to_string())
print("\non seeded entries")
print(ev.loc[(F == 1) & (C == 1), "tm_status"].value_counts().to_string())

## Step 7: provisional adjustment, F ~ C + T on the rows where T exists

This adjustment remains provisional. The export still awaits Pierre's confirmation, rolling topic fit may partly capture a mediator, and complete cases form a selected subset of the full table. These estimates are useful for orientation, but are not yet final report results.

- logistic regression, standard errors clustered by author
- the adjusted ratio comes from predicting every included row once with `C = 1` and once with `C = 0`, then dividing the two mean predicted risks
- overlap is checked first, adjustment only means something where seeded and unseeded rows share the same range of `T`

In [ ]:
m = has
X = np.column_stack([np.ones(m.sum()), C[m].astype(float), ev["T"].values[m]])
y = F[m].astype(float)
print(f"complete cases {int(m.sum()):,} rows, {int(y.sum()):,} entries")
print(f"naive ratio inside this population {(y[X[:,1]==1].mean())/(y[X[:,1]==0].mean()):.2f}, "
      f"against 6.32 in the full table, so the population shift is real\n")

# overlap in T between the two exposure groups
q = [0.05, 0.25, 0.5, 0.75, 0.95]
qt1 = np.quantile(X[X[:, 1] == 1, 2], q)
qt0 = np.quantile(X[X[:, 1] == 0, 2], q)
print("T quantiles 5/25/50/75/95")
print("  with seed    " + "  ".join(f"{v:.2f}" for v in qt1))
print("  without      " + "  ".join(f"{v:.2f}" for v in qt0))
print()

beta = np.zeros(3)
for _ in range(25):   # newton, converges in a handful of steps on this size
    p = 1 / (1 + np.exp(-(X @ beta)))
    H = (X * (p * (1 - p))[:, None]).T @ X
    step = np.linalg.solve(H, X.T @ (y - p))
    beta += step
    if np.abs(step).max() < 1e-10:
        break

# sandwich variance with author clusters
p = 1 / (1 + np.exp(-(X @ beta)))
cl = codes[m]
U = X * (y - p)[:, None]
order = np.argsort(cl)
S = np.zeros((3, 3))
for blk in np.split(U[order], np.flatnonzero(np.diff(cl[order])) + 1):
    s = blk.sum(0)
    S += np.outer(s, s)
Hinv = np.linalg.inv((X * (p * (1 - p))[:, None]).T @ X)
se = np.sqrt(np.diag(Hinv @ S @ Hinv))

print(f"C   log odds {beta[1]:.3f}, OR {np.exp(beta[1]):.2f}, "
      f"cluster 95% CI [{np.exp(beta[1]-1.96*se[1]):.2f}, {np.exp(beta[1]+1.96*se[1]):.2f}]")
print(f"T   log odds {beta[2]:.3f}, per 0.1 of topic fit OR {np.exp(beta[2]/10):.2f}")

X1 = np.column_stack([X[:, 0], np.ones(len(X)), X[:, 2]])
X0 = np.column_stack([X[:, 0], np.zeros(len(X)), X[:, 2]])

def log_rr(b):
    p1 = 1 / (1 + np.exp(-(X1 @ b)))
    p0 = 1 / (1 + np.exp(-(X0 @ b)))
    return np.log(p1.mean() / p0.mean())

# delta method, the ratio is a smooth function of beta so its variance follows from the sandwich
grad = np.zeros(3)
for k in range(3):
    d = np.zeros(3); d[k] = 1e-6
    grad[k] = (log_rr(beta + d) - log_rr(beta - d)) / 2e-6
se_lrr = float(np.sqrt(grad @ (Hinv @ S @ Hinv) @ grad))
rr = np.exp(log_rr(beta))
print(f"\nadjusted rate ratio, g computation  {rr:.2f}, "
      f"cluster 95% CI [{rr*np.exp(-1.96*se_lrr):.2f}, {rr*np.exp(1.96*se_lrr):.2f}]")

# Q3_ind adjusted, the outcome is entry without the seed co-author on the paper, under C = 0 that is every entry
yind = ((F[m] == 1) & (ev["first_entry_ride"].values[m] == 0)).astype(float)
bi = np.zeros(3)
for _ in range(25):
    pi_ = 1 / (1 + np.exp(-(X @ bi)))
    step = np.linalg.solve((X * (pi_ * (1 - pi_))[:, None]).T @ X, X.T @ (yind - pi_))
    bi += step
    if np.abs(step).max() < 1e-10:
        break
pi_ = 1 / (1 + np.exp(-(X @ bi)))
Ui = X * (yind - pi_)[:, None]
Si = np.zeros((3, 3))
for blk in np.split(Ui[order], np.flatnonzero(np.diff(cl[order])) + 1):
    s_ = blk.sum(0)
    Si += np.outer(s_, s_)
Hi = np.linalg.inv((X * (pi_ * (1 - pi_))[:, None]).T @ X)

def log_rr_ind(b):
    return np.log((1 / (1 + np.exp(-(X1 @ b)))).mean() / (1 / (1 + np.exp(-(X0 @ b)))).mean())

gi = np.zeros(3)
for k in range(3):
    d = np.zeros(3); d[k] = 1e-6
    gi[k] = (log_rr_ind(bi + d) - log_rr_ind(bi - d)) / 2e-6
se_i = float(np.sqrt(gi @ (Hi @ Si @ Hi) @ gi))
rri = np.exp(log_rr_ind(bi))
print(f"Q3_ind adjusted, g computation      {rri:.2f}, cluster 95% CI [{rri*np.exp(-1.96*se_i):.2f}, {rri*np.exp(1.96*se_i):.2f}]")

## Step 8: where the change in the adjusted ratio comes from

The thresholded and threshold-free runs do not measure exactly the same `T`: the journal profiles change with the included papers, and the kernel scale is re-estimated. The same model is therefore run three times: first with the old values on the shared rows, then with the new values on those rows, and finally with all newly covered rows. This is a diagnostic sequence, not a unique attribution, because its order affects the intermediate result.

In [ ]:
# the thresholded run, values on the same rows for the value-swap comparison
tmin3 = pd.read_csv("../data/event_table_topicmatch_local_min3.csv", low_memory=False,
                    usecols=["author_id", "journal_id", "t", "topic_match"], dtype={"t": "int16"})
assert (tmin3["author_id"].values == ev["author_id"].values).all()   # same rows in the same order
assert (tmin3["journal_id"].values == ev["journal_id"].values).all()
assert (tmin3["t"].values == ev["t"].values).all()
told = tmin3["topic_match"].values
del tmin3

def adjusted_rr(mask, T):
    Xd = np.column_stack([np.ones(mask.sum()), C[mask].astype(float), T[mask]])
    yd = F[mask].astype(float)
    b = np.zeros(3)
    for _ in range(30):
        pd_ = 1 / (1 + np.exp(-(Xd @ b)))
        step = np.linalg.solve((Xd * (pd_ * (1 - pd_))[:, None]).T @ Xd, Xd.T @ (yd - pd_))
        b += step
        if np.abs(step).max() < 1e-10:
            break
    p1 = (1 / (1 + np.exp(-(np.column_stack([Xd[:, 0], np.ones(len(Xd)), Xd[:, 2]]) @ b)))).mean()
    p0 = (1 / (1 + np.exp(-(np.column_stack([Xd[:, 0], np.zeros(len(Xd)), Xd[:, 2]]) @ b)))).mean()
    return p1 / p0

tnew = ev["T"].values
shared = ~np.isnan(told) & ~np.isnan(tnew)
print(f"kernel scale, thresholded 0.3613 against threshold-free 0.3611, so nearly unchanged")
print(f"shared rows {shared.sum():,}")
print(f"  old values, shared rows        {adjusted_rr(shared, told):.2f}")
print(f"  new values, same shared rows   {adjusted_rr(shared, tnew):.2f}")
print(f"  new values, full new rows      {adjusted_rr(~np.isnan(tnew), tnew):.2f}")

## Step 9: journal and year effects

The models above hold topic fit constant and nothing else. Journals differ in how often anyone enters them, and years differ too, and connections concentrate where entry rates are high, so both are plausible confounders of `C` and `F`. One journal has 17,936 measurable-T rows and no entry at all, its unpenalized coefficient has no finite maximum, so it is dropped here and the C + T baseline is refit on exactly the remaining rows. That exclusion is conditioned on the outcome and is stated, not hidden. This step is a sensitivity until the team decides the main specification.

In [ ]:
# rows with T, minus the journal that never sees an entry there
mfe = m.copy()
jent = ev.loc[m].assign(Fv=F[m]).groupby("journal_id", observed=True)["Fv"].sum()
dropped = jent[jent == 0].index.tolist()
mfe &= ~ev["journal_id"].isin(dropped).values
print(f"journals without any entry among measurable-T rows: {dropped}, {int(m.sum() - mfe.sum()):,} rows dropped")
jc, jl = pd.factorize(ev.loc[mfe, "journal_id"]); yc, yl = pd.factorize(ev.loc[mfe, "t"])
print(f"{int(mfe.sum()):,} rows, {len(jl)} journals, {len(yl)} years")

def design_fe(cvals):
    n = int(mfe.sum()); Xf = np.zeros((n, 3 + len(jl) - 1 + len(yl) - 1), dtype=np.float32)
    Xf[:, 0] = 1; Xf[:, 1] = cvals; Xf[:, 2] = ev["T"].values[mfe]
    for j in range(1, len(jl)): Xf[jc == j, 2 + j] = 1
    for y_ in range(1, len(yl)): Xf[yc == y_, 1 + len(jl) + y_] = 1
    return Xf

def fit_rr(Xf, yv):
    b = np.zeros(Xf.shape[1])
    for _ in range(40):
        p_ = 1 / (1 + np.exp(-(Xf @ b)))
        step = np.linalg.solve((Xf * (p_ * (1 - p_))[:, None]).T @ Xf + 1e-8 * np.eye(len(b)), Xf.T @ (yv - p_))
        b += step
        if np.abs(step).max() < 1e-8: break
    X1_ = Xf.copy(); X1_[:, 1] = 1; X0_ = Xf.copy(); X0_[:, 1] = 0
    return (1 / (1 + np.exp(-(X1_ @ b)))).mean() / (1 / (1 + np.exp(-(X0_ @ b)))).mean()

Xfe = design_fe(C[mfe].astype(np.float32))
yall = F[mfe].astype(np.float32)
yind = ((F[mfe] == 1) & (ev["first_entry_ride"].values[mfe] == 0)).astype(np.float32)
for name, yv in (("Q3_all", yall), ("Q3_ind", yind)):
    print(f"{name}   C + T on these rows {fit_rr(Xfe[:, :3], yv):.2f}   C + T + journal + year {fit_rr(Xfe, yv):.2f}")

## Summary

| number | value | status |
|---|---|---|
| crude rate ratio, full table | 6.32, CI 6.04 to 6.59 | solid as a crude association |
| seeded entries | 1,784, of which 756 rides and 1,028 independent | confirmed by independent Python and Prolog implementations |
| yearly contrast | 5.46 to 6.93 from 2021 | descriptive, no stability claim |
| T coverage | 17.2% of rows, 99.0% of seeded entries | Pierre's v5 export |
| Q3_ind, crude | 3.64, CI 3.42 to 3.85 | pre-registered headline form |
| Q3_ind, topic-adjusted | 3.34, CI 3.12 to 3.58 | Pierre's v5 export, confirmed by an independent regeneration |
| crude ratio, complete cases | 10.16 | shows the population shift |
| topic-adjusted ratio, Q3_all | 6.09, CI 5.76 to 6.45 | Pierre's v5 export, confirmed by an independent regeneration |

Removing the whole-period threshold changed the adjusted ratio from 5.24 to 6.09. On the shared rows, replacing the old topic values with the new ones changes it from 5.24 to 5.69. Adding the newly covered rows changes it further to 6.09. This sequence is descriptive and depends on the order of the steps. In the threshold-free complete cases, `T` has a positive adjusted association with entry (OR 2.40 per 0.1 increase in topic fit). Step 9 changes the picture: with journal and year effects the adjusted ratio falls from 6.01 to 2.27 for Q3_all and from 3.30 to 1.18 for Q3_ind on the same rows, so most of the association above is between-journal and between-year composition. Which specification is the main one is the open methods decision, and 6.09 should not be shown without 2.26 next to it.

Q3_ind is the pre-registered headline form (decisions log, 2026-08-08): entries without the seed co-author on the entering paper, against the unseeded entry rate. It is lower than Q3_all by construction, the rides leave the numerator, and it stays well above one, so seeded authors enter more often even when the seed co-author is not on the paper.

Still open, in order

1. Pierre confirms the threshold-free export or reruns it himself
2. Pierre decides whether to keep the current embeddings or rerun with the SPECTER2 adapter confirmed active
3. the group decides how to handle missing `T`; observed values remain continuous and `tm_status` records why a value is missing
4. decide the main specification, T only as pre-registered or with journal and year effects, and justify it in writing, then run the agreed final model on the v5 export